# 02 · Entrenamiento con LoRA

Entrena las **configs A y B** (difieren solo en `target_modules`: atención vs atención + MLP;
`r=16, alpha=32`, `lr=2e-4`, 1 época, batch efectivo 16, semilla 2026).
Cada corrida en la GPU T4 toma **35–50 min** (y la B algo más).

In [ ]:
# --- Preparar el entorno como en el notebook 01 ---
import os
REPO = "USUARIO/proyecto3-text2sql"   # <-- cambiar por el repo de la pareja
if not os.path.exists("/content/proyecto3-text2sql"):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    os.system(f"git clone https://{token}@github.com/{REPO} /content/proyecto3-text2sql")
os.chdir("/content/proyecto3-text2sql")
!pip install -q -r requirements.txt | tail -1
!nvidia-smi | head -12
import os as _os
assert _os.path.exists('solucion.py')


## 1. Si el notebook 01 no corrió en esta sesión: MySQL + esquemas + dataset

In [ ]:
import subprocess, glob
if not glob.glob('datos/esquemas/train/*.sql'):
    !bash scripts/instalar_mysql_colab.sh
    !python datos/construir_esquemas.py --socket /tmp/mysql.sock --crear-lector
    !python datos/validar_pares.py --socket /tmp/mysql.sock
    !python datos/generar_reparaciones.py
print('Esquemas y pares listos')


## 2. Dataset en formato de chat (mismo prompt que en inferencia)

In [ ]:
!python src/datos_chat.py
import pandas as pd
import json, os
ds = pd.read_json('datos/entrenamiento/dataset_chat.jsonl', lines=True)
display(ds.groupby(['tipo','db']).size().unstack(fill_value=0))
ejemplo = json.loads(open('datos/entrenamiento/dataset_chat.jsonl').readline())
print('--- assistant ---'); print(ejemplo['mensajes'][-1]['content'][:200])


## 3. Hipótesis (escritas antes de entrenar, en `resultados/hipotesis.md`)

In [ ]:
print(open('resultados/hipotesis.md').read()[:1800])


## 4. Entrenar config A — solo atención (35–50 min en T4)

In [ ]:
!python src/entrenar.py --config A

In [ ]:
import json
rA = json.load(open('resultados/entrenamiento_A.json'))
print('A · entrenables:', rA['parametros_entrenables'], 'min:', rA['minutos_entrenamiento'],
      '· eval_loss_final:', round(rA['eval_loss_final'], 4),
      f"({rA['porcentaje_entrenable']:.3f}% del total)")


## 5. Entrenar config B — atención + MLP (~4× params, 45–60 min en T4)

In [ ]:
!python src/entrenar.py --config B

In [ ]:
rB = json.load(open('resultados/entrenamiento_B.json'))
print('B · entrenables:', rB['parametros_entrenables'], 'min:', rB['minutos_entrenamiento'],
      '· eval_loss_final:', round(rB['eval_loss_final'], 4),
      f"({rB['porcentaje_entrenable']:.3f}% del total)")


## 6. Curvas de loss y comparación

In [ ]:
from IPython.display import Image
Image('resultados/loss_A.png')


In [ ]:
Image('resultados/loss_B.png')


In [ ]:
import json
A = json.load(open('resultados/entrenamiento_A.json'))
B = json.load(open('resultados/entrenamiento_B.json'))
print('      A    B')
print('eval loss  %.3f  %.3f' % (A['eval_loss_final'], B['eval_loss_final']))
print('min T4     %s    %s' % (A['minutos_entrenamiento'], B['minutos_entrenamiento']))
print('parámetros entrenables: {:,} vs {:,}'.format(A['parametros_entrenables'], B['parametros_entrenables']))


> Los EX de A y B se deciden en el notebook 03 (validación propia). De aquí sales las curvas,
> los parámetros entrenables y los tiempos, que son de lo que reporta `entrenamiento_{A,B}.json`.